# M3 baselines (B0-B5), the C1 few-shot curve and K1 enrolment

What each model gives on our validation photos before any training on them (plan section 5.2). Each baseline cell
writes `/kaggle/working/baselines/<B-id>/final_metrics.json` in the training runner's shape (head, safety,
calibration, taxonomy, run), its score on validation photos uploaded from 2022 (`upload_2022_slice`),
`versions.json`, and `predictions.npz` where there are per-photo predictions. The last cell prints one comparison
table.

- **B0** class frequencies of the capped subset. **B1** Bolon et al. 2022's global ViT (ONNX from LINDAT), labels
  mapped to our classes; full validation is `upper_bound_overlap_unverified`, the 2022 slice is the fair score.
  **B2** BioCLIP zero-shot from taxonomic prompts; its training data runs past 2022, so even the slice is "overlap not
  verified". **B3** the three E1 encoders, frozen ImageNet weights, nearest prototype from the subset. **B4** DINOv2
  ViT-S/14 frozen at 322 px, prototypes and a linear probe. **B5** YOLOv8n trained on Open Images V7, as is, on the
  Open Images snake boxes. **B6** is not run (ruling R22).
- **C1** prototypes from 5, 10 and 20 photos of distinct observations on tier 2 classes, on DINOv2 features here and
  on E2 features once an E2 run exists. **K1** enrols the K1 run's ten held-out species from 1, 5 and 10 photos.

Code: `../src` and `../configs/m3` (the runner kernel unpacks them into `/kaggle/working/repo`). Data inputs are found
with `kagglehub.dataset_download`: `serpentguard-indian-snakes-clean`, `serpentguard-sam-masks`,
`serpentguard-m3-subset` and `serpentguard-extended-sources`. Internet must be on (pip, LINDAT, Hugging Face,
Ultralytics weights).

In [1]:
RUN_BASELINES = ['B1', 'B3']
LIMIT = None       # first N photos of each frame (and N Open Images photos), for a smoke run
E2_RUN_DIR = None  # an E2 run's output folder (val_outputs.npz, calib_outputs.npz): C1 on E2 features
K1_RUN_DIR = None  # the K1 run's output folder (heldout_outputs.npz, prototypes.npz, val_outputs.npz)

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the per-transform augmentation seeding was verified on it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}")

albumentations 2.0.8, timm 1.0.31


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: B4's probe and every metric use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
OPEN_CLIP_VERSION = "2.32.0"  # BioCLIP (B2) loads through open_clip; timm stays at the pin above
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"open_clip_torch=={OPEN_CLIP_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)

import open_clip

assert open_clip.__version__ == OPEN_CLIP_VERSION, f"open_clip {open_clip.__version__} is loaded"
EXTRA_VERSIONS = {"open_clip_torch": open_clip.__version__}
print(f"open_clip {open_clip.__version__}")

open_clip 2.32.0


In [5]:
ONNXRUNTIME_VERSION = "1.31.1"  # requirements-m3.txt's onnxruntime, as the GPU build: B1's ViT is 1.13 GB
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime"], check=True)  # one build only
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"onnxruntime-gpu=={ONNXRUNTIME_VERSION}"], check=True)

import onnxruntime

assert onnxruntime.__version__ == ONNXRUNTIME_VERSION, f"onnxruntime {onnxruntime.__version__} is loaded"
EXTRA_VERSIONS["onnxruntime-gpu"] = onnxruntime.__version__
print(f"onnxruntime {onnxruntime.__version__}, providers {onnxruntime.get_available_providers()}")

onnxruntime 1.31.1, providers ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [6]:
ULTRALYTICS_VERSION = "8.4.202"  # requirements-m3.txt; B5 runs yolov8n-oiv7.pt through it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

import ultralytics

assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
EXTRA_VERSIONS["ultralytics"] = ultralytics.__version__
print(f"ultralytics {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.202


In [7]:
import hashlib
import json
import shutil
import time
import urllib.request
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))
CONFIGS = CODE.parent / "configs" / "m3"

import numpy as np
import pandas as pd
import torch

from serpentguard.baselines import (BIOCLIP_HUB, BIOCLIP_LINEAGE, BIOCLIP_TEMPLATES, BOLON_ID_COLUMN,
                                    BOLON_NAME_COLUMN, DINOV2_INPUT_SIZE, OPEN_IMAGES_EVALUATION, OVERLAP_CLEARED,
                                    OVERLAP_NOT_VERIFIED, PROBE_C, PROBE_MAX_ITER, PROTOTYPE_SCALE,
                                    SLICE_UPLOADED_FROM, SYNONYMS, UPPER_BOUND, YOLO_ALL_CONF, YOLO_BATCH, YOLO_CONF,
                                    YOLO_WEIGHTS, OnnxClassifier, baseline_metrics, bioclip_baseline, bolon_baseline,
                                    check_label_order, detection_baseline_scores, embed_frozen,
                                    frozen_encoder, frozen_prototype_baseline, frozen_run_record,
                                    labels_from_metadata, linear_probe, logit_scale,
                                    map_labels, open_images_ground_truth, open_onnx, photo_share, prior_baseline,
                                    save_baseline, snake_class_index, summary_row, taxonomic_names, upload_2022_slice,
                                    yolo_snake_boxes)
from serpentguard.config import load_config
from serpentguard.fewshot import enrolment_eval, fewshot_curve, observation_ids
from serpentguard.metrics import DET_IOU
from serpentguard.models.encoders import DINOV2, INPUT_SIZE, MOBILE_ENCODERS
from serpentguard.run_data import read_tables, run_frames
from serpentguard.run_outputs import class_tables, subset_hash, write_json

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
MASKS = input_root("serpentguard-sam-masks")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
EXTENDED = input_root("serpentguard-extended-sources")
OUT = Path("/kaggle/working/baselines")
FEWSHOT_OUT = Path("/kaggle/working/fewshot")
DOWNLOADS = Path("/kaggle/tmp/baselines")  # outside /kaggle/working, so the 1.13 GB model isn't saved as output
FRAME_CONFIG = "E1_mnv4"  # F0, whole photo: the runner's capped subset and full validation, as every E1 run sees them
ENCODER_IDS = {"mobilenetv4_conv_small": "mnv4", "efficientnet_b0": "effb0", "mobilevit_s": "mvits"}  # E1's suffixes
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256
EXPECTED_SLICE_PHOTOS = 13_947         # validation photos uploaded from 2022 (plan section 3)
TIER2 = "tier2"
BATCH = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKERS = os.cpu_count() or 2

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", SUBSET,
                 EXTENDED / "open_images_snake_boxes.csv", EXTENDED / "open_images_snake_masks.csv"):
    assert required.exists(), f"missing input {required}"
assert set(ENCODER_IDS) == set(MOBILE_ENCODERS), "B3 must cover exactly the three E1 encoders"

cfg = load_config(CONFIGS / f"{FRAME_CONFIG}.yaml")
tables = read_tables(cfg, DATA, MASKS, SUBSET)
frames = run_frames(cfg, tables, DATA, MASKS, limit=LIMIT)
taxa = class_tables(tables.classes)
train, val, calibration = frames.train, frames.val, frames.calibration
train_counts = np.bincount(train["class_id"], minlength=taxa.n_classes)
in_2022 = upload_2022_slice(val, tables.metadata)
digest = subset_hash(tables.subset["path"])
assert digest.startswith(EXPECTED_SUBSET_HASH), f"another subset was read: {digest}"
if LIMIT is None:
    assert in_2022.sum() == EXPECTED_SLICE_PHOTOS, f"{in_2022.sum()} validation photos are uploaded from 2022"
RUN = {"subset_hash": digest, "train_photos": len(train), "device": str(DEVICE), "limit": LIMIT,
       "config": {"frame_config": FRAME_CONFIG, "front_end": cfg.front_end, "frame_input_size": cfg.input_size,
                  "slice_uploaded_from": SLICE_UPLOADED_FROM, "batch": BATCH, "workers": WORKERS,
                  "subset": str(SUBSET), "expected_subset_hash": EXPECTED_SUBSET_HASH}}
OUT.mkdir(parents=True, exist_ok=True)
FEWSHOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"code {CODE}; {len(train)} training photos after caps, {len(val)} validation ({int(in_2022.sum())} uploaded "
      f"from 2022, {val['class_id'][in_2022].nunique()} classes), {len(calibration)} calibration; {DEVICE}, "
      f"{WORKERS} workers")


def save(baseline_id, metrics, outputs=None, frame=val, extra_versions=None):
    saved = save_baseline(OUT / baseline_id, metrics, {**EXTRA_VERSIONS, **(extra_versions or {})},
                          frame if outputs else None, outputs)
    print({key: value for key, value in summary_row(baseline_id, saved).items() if value == value})
    return saved

code /kaggle/working/repo/src; 13611 training photos after caps, 17151 validation (13947 uploaded from 2022, 174 classes), 7999 calibration; cuda, 4 workers


## B1 download

The model and its label list come from the LINDAT item `20.500.12800/1-4773` (BSD-3). The file URLs and MD5s were
read from the item page saved on 22 Oct 2026; the download checks each MD5. The labels are the SnakeCLEF 2021
training metadata's `class_id` -> `binomial` pairs. The cell prints the model's input and output, so check them
before trusting B1: the graph's name says it preprocesses its own input.

In [8]:
LINDAT = "https://lindat.mff.cuni.cz/repository/server/api/core/bitstreams/handle/20.500.12800/1-4773"
BOLON_MODEL_URL = f"{LINDAT}/ViT_PreProcessing-ops11-preprocessing-int-dynam_graph.onnx?sequence=7&isAllowed=y"
BOLON_MODEL_MD5 = "838ae0f8ed19ffeee89aba5fa8b50956"
BOLON_LABELS_URL = f"{LINDAT}/min-train_metadata.csv?sequence=2&isAllowed=y"
BOLON_LABELS_MD5 = "52ec43c8595a3835da6aa16364aff75e"
BOLON_INPUT_SIZE = None        # read from the model; set (height, width) only if its input size is dynamic
BOLON_MEAN = BOLON_STD = None  # 0-255 pixels go in; set ImageNet values only if the input is a normalised float
DOWNLOAD_TIMEOUT_SECONDS, CHUNK_BYTES = 120, 2 ** 24
LABEL_SAMPLE = 8


def download(url, path, md5):
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        with urllib.request.urlopen(url, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response, open(partial, "wb") as out:
            shutil.copyfileobj(response, out, CHUNK_BYTES)
        partial.rename(path)
    digest = hashlib.md5()
    with open(path, "rb") as stored:
        for chunk in iter(lambda: stored.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    assert digest.hexdigest() == md5, f"{path.name}: MD5 {digest.hexdigest()}, expected {md5}"
    return path


if "B1" in RUN_BASELINES:
    bolon_path = download(BOLON_MODEL_URL, DOWNLOADS / "bolon_vit.onnx", BOLON_MODEL_MD5)
    bolon_session = open_onnx(bolon_path)
    for spec in bolon_session.get_inputs() + bolon_session.get_outputs():
        print(f"{spec.name}: shape {spec.shape}, {spec.type}")
    bolon_outputs = bolon_session.get_outputs()[0].shape[-1]
    assert isinstance(bolon_outputs, int), f"the model's output width is {bolon_outputs!r}, not a fixed number"
    bolon_labels = labels_from_metadata(pd.read_csv(  # an id with no photo in the metadata leaves a None slot (R31)
        download(BOLON_LABELS_URL, DOWNLOADS / "snakeclef2021_min_train_metadata.csv", BOLON_LABELS_MD5),
        low_memory=False), bolon_outputs)
    bolon_unnamed = sum(name is None for name in bolon_labels)
    print(f"providers {bolon_session.get_providers()}; {bolon_outputs} outputs, {bolon_unnamed} without a name in "
          f"the metadata; e.g. {[name for name in bolon_labels if name is not None][:LABEL_SAMPLE]}")
    bolon = OnnxClassifier(bolon_path, bolon_labels, BOLON_INPUT_SIZE, BOLON_MEAN, BOLON_STD, session=bolon_session)
    _, label_coverage = map_labels(bolon_labels, taxa.species)
    print(f"input {bolon.input_size}, {'uint8' if bolon.integer_input else 'float'}, "
          f"{'NCHW' if bolon.channels_first else 'NHWC'}; covers {label_coverage['classes_covered']} of our "
          f"{label_coverage['classes']} classes; renamed {label_coverage['renamed']}")

input: shape ['?', '?', 3], tensor(int32)
output: shape [1, 772], tensor(float)
2404: shape [1, 772], tensor(float)
providers ['CUDAExecutionProvider', 'CPUExecutionProvider']; 772 outputs, 4 without a name in the metadata; e.g. ['Acanthophis antarcticus', 'Acanthophis hawkei', 'Acanthophis praelongus', 'Acanthophis pyrrhus', 'Acanthophis rugosus', 'Acanthophis wellsi', 'Achalinus formosanus', 'Achalinus niger']


ValueError: can't tell the channel axis of input shape ['?', '?', 3]

In [9]:
if "B0" in RUN_BASELINES:
    probs = prior_baseline(train_counts, len(val))
    save("B0", baseline_metrics(val, probs, train_counts, taxa, in_2022,
                                {**RUN, "run_id": "B0", "baseline": "class frequencies of the capped subset"}))

In [10]:
B1_CHECK_PHOTOS = 256

if "B1" in RUN_BASELINES:
    covered = val[val["class_id"].isin(label_coverage["covered_class_ids"])]
    covered = covered.sample(min(B1_CHECK_PHOTOS, len(covered)), random_state=42)
    check_probs, _ = bolon_baseline(bolon, covered, taxa.species, batch_size=BATCH, num_workers=WORKERS)
    b1_check = check_label_order(check_probs, covered["class_id"].to_numpy(), label_coverage["covered_class_ids"])
    print(f"B1 top-1 on {b1_check['photos']} validation photos of covered classes: {b1_check['top1']:.3f} "
          f"(needs {b1_check['threshold']:.3f}; chance {b1_check['chance']:.4f})")

NameError: name 'label_coverage' is not defined

In [11]:
if "B1" in RUN_BASELINES:
    started = time.perf_counter()
    probs, coverage = bolon_baseline(bolon, val, taxa.species, batch_size=BATCH, num_workers=WORKERS)
    coverage["photo_share_upload_2022_slice"] = photo_share(val["class_id"][in_2022], coverage["covered_class_ids"])
    run = {**RUN, "run_id": "B1", "baseline": "Bolon et al. 2022 global ViT (ONNX, LINDAT 20.500.12800/1-4773)",
           "onnx": bolon.describe(), "model_url": BOLON_MODEL_URL, "model_md5": BOLON_MODEL_MD5,
           "labels_url": BOLON_LABELS_URL, "labels_md5": BOLON_LABELS_MD5,
           "label_columns": {"id": BOLON_ID_COLUMN, "name": BOLON_NAME_COLUMN}, "synonyms": SYNONYMS,
           "unnamed_outputs": bolon_unnamed, "label_order_check": b1_check,
           "seconds": time.perf_counter() - started}
    save("B1", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, evaluation=UPPER_BOUND,
                                slice_overlap=OVERLAP_CLEARED, extra={"coverage": coverage}),
         {"probs": probs.astype(np.float32)}, extra_versions={"bolon_model_md5": BOLON_MODEL_MD5})

NameError: name 'bolon' is not defined

In [12]:
if "B2" in RUN_BASELINES:
    started = time.perf_counter()
    bioclip, _, bioclip_preprocess = open_clip.create_model_and_transforms(BIOCLIP_HUB)
    class_names = taxonomic_names(tables.classes)
    print(f"prompts like {BIOCLIP_TEMPLATES[0].format(class_names[0])!r}")
    probs = bioclip_baseline(bioclip.to(DEVICE), open_clip.get_tokenizer(BIOCLIP_HUB), bioclip_preprocess, val,
                             class_names, BIOCLIP_TEMPLATES, batch_size=BATCH, num_workers=WORKERS, device=DEVICE)
    run = {**RUN, "run_id": "B2", "baseline": f"BioCLIP zero-shot ({BIOCLIP_HUB}), taxonomic prompts",
           "hub": BIOCLIP_HUB, "templates": list(BIOCLIP_TEMPLATES), "lineage": BIOCLIP_LINEAGE,
           "logit_scale": logit_scale(bioclip), "preprocess": repr(bioclip_preprocess),
           "seconds": time.perf_counter() - started}
    save("B2", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, evaluation=UPPER_BOUND,
                                slice_overlap=OVERLAP_NOT_VERIFIED), {"probs": probs.astype(np.float32)})
    del bioclip
    torch.cuda.empty_cache()

In [13]:
if "B3" in RUN_BASELINES:
    for encoder_name, suffix in ENCODER_IDS.items():
        started = time.perf_counter()
        encoder = frozen_encoder(encoder_name)
        features_train, features_val = (embed_frozen(encoder_name, frame, batch_size=BATCH, num_workers=WORKERS,
                                                     device=DEVICE, encoder=encoder) for frame in (train, val))
        probs = frozen_prototype_baseline(features_train, train["class_id"], features_val, n_classes=taxa.n_classes)
        run = {**RUN, "run_id": f"B3_{suffix}", "baseline": f"{encoder_name}, frozen ImageNet weights",
               "head_is": "nearest prototype from the subset, softmax(prototype_scale x cosine)",
               **frozen_run_record(encoder_name, encoder, INPUT_SIZE), "seconds": time.perf_counter() - started}
        save(f"B3_{suffix}", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, prototype_probs=probs),
             {"probs": probs.astype(np.float32)})

model.safetensors: reconstructing file:   0%|          |  0.00B / 15.2MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_mnv4', 'evaluation': 'full_validation', 'macro_f1': 0.09239042613103067, 'macro_f1_prototype': 0.09239042613103067, 'top1': 0.156608944084893, 'top3': 0.2978835053349659, 'who_recall': 0.328484279680901, 'o3_raw': 0.671515720319099, 'ece': 0.037257473830600844, 'macro_f1_2022': 0.09065020996875518, 'top1_2022': 0.15551731555173157, 'slice_overlap': 'no labelled photos from outside our training subset'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_effb0', 'evaluation': 'full_validation', 'macro_f1': 0.14001803996856246, 'macro_f1_prototype': 0.14001803996856246, 'top1': 0.2287330184828873, 'top3': 0.40767302198122557, 'who_recall': 0.4514312529328954, 'o3_raw': 0.5485687470671047, 'ece': 0.039028885938134014, 'macro_f1_2022': 0.13524535755923214, 'top1_2022': 0.22528142252814226, 'slice_overlap': 'no labelled photos from outside our training subset'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_mvits', 'evaluation': 'full_validation', 'macro_f1': 0.13520903996764572, 'macro_f1_prototype': 0.13520903996764572, 'top1': 0.2171885021281558, 'top3': 0.4001515946592035, 'who_recall': 0.44251525105584233, 'o3_raw': 0.5574847489441577, 'ece': 0.03226574227012736, 'macro_f1_2022': 0.1319647582534319, 'top1_2022': 0.21488492148849214, 'slice_overlap': 'no labelled photos from outside our training subset'}


In [14]:
if "B4" in RUN_BASELINES or "C1" in RUN_BASELINES:
    started = time.perf_counter()
    dino_encoder = frozen_encoder(DINOV2, DINOV2_INPUT_SIZE)
    dino = {name: embed_frozen(DINOV2, frame, input_size=DINOV2_INPUT_SIZE, batch_size=BATCH, num_workers=WORKERS,
                               device=DEVICE, encoder=dino_encoder)
            for name, frame in (("train", train), ("val", val), ("calibration", calibration))}
    embed_seconds = time.perf_counter() - started
if "B4" in RUN_BASELINES:
    probe = linear_probe(dino["train"], train["class_id"], dino["val"], n_classes=taxa.n_classes)
    prototype = frozen_prototype_baseline(dino["train"], train["class_id"], dino["val"], n_classes=taxa.n_classes)
    run = {**RUN, "run_id": "B4", "baseline": f"{DINOV2} frozen at {DINOV2_INPUT_SIZE} px",
           "head_is": "linear probe: multinomial logistic regression (lbfgs) on L2-normalised features",
           **frozen_run_record(DINOV2, dino_encoder, DINOV2_INPUT_SIZE), "probe_c": PROBE_C,
           "probe_max_iter": PROBE_MAX_ITER, "probe_class_weight": "balanced", "embed_seconds": embed_seconds,
           "seconds": time.perf_counter() - started}
    save("B4", baseline_metrics(val, probe, train_counts, taxa, in_2022, run, prototype_probs=prototype),
         {"probs": probe.astype(np.float32), "prototype_probs": prototype.astype(np.float32)})
    np.savez_compressed(OUT / "B4" / "features.npz", **{f"{name}_features": value for name, value in dino.items()},
                        **{f"{name}_paths": frame["path"].to_numpy(dtype=str) for name, frame in
                           (("train", train), ("val", val), ("calibration", calibration))})

In [15]:
def tier2_curve(features, frame, description):
    curve = fewshot_curve(features, frame["class_id"], observation_ids(frame["path"], tables.metadata),
                          np.flatnonzero(taxa.tier == TIER2))
    curve["features"], curve["pool"] = description, "validation and calibration photos (never trained on)"
    for shots, scores in curve["shots"].items():
        print(f"{description}, {shots} shots: top-1 {scores['top1']:.3f} +/- {scores['top1_std']:.3f}, "
              f"per-class {scores['macro_top1']:.3f} on {len(curve['classes'])} tier 2 classes")
    return curve


if "C1" in RUN_BASELINES:
    pool = pd.concat([val, calibration], ignore_index=True)
    write_json(FEWSHOT_OUT / "C1_dinov2.json",
               tier2_curve(np.concatenate([dino["val"], dino["calibration"]]), pool,
                           f"{DINOV2} frozen, {DINOV2_INPUT_SIZE} px"))

In [16]:
if "B5" in RUN_BASELINES:
    from ultralytics import YOLO

    started = time.perf_counter()
    detector = YOLO(YOLO_WEIGHTS)
    images, truth = open_images_ground_truth(pd.read_csv(EXTENDED / "open_images_snake_boxes.csv"),
                                             pd.read_csv(EXTENDED / "open_images_snake_masks.csv"))
    if LIMIT is not None:
        images, truth = images.head(LIMIT), truth[:LIMIT]
    boxes = yolo_snake_boxes(detector, [str(EXTENDED / name) for name in images["image_file"]], normalized=True)
    run = {**RUN, "run_id": "B5", "baseline": f"{YOLO_WEIGHTS} (Open Images V7), as is",
           "evaluation": OPEN_IMAGES_EVALUATION, "weights": YOLO_WEIGHTS,
           "snake_class": snake_class_index(detector.names),
           "conf": YOLO_CONF, "all_conf": YOLO_ALL_CONF, "iou_threshold": DET_IOU, "batch": YOLO_BATCH,
           "images": len(images), "rotated_images": int(images["rotation"].fillna(0).ne(0).sum()),
           "boxes": "normalised to the photo; IoU doesn't change when each axis is scaled",
           "seconds": time.perf_counter() - started}
    save("B5", {"detection": detection_baseline_scores(boxes, truth), "run": run})
    write_json(OUT / "B5" / "predictions.json", {"image_id": images["image_id"].tolist(), "boxes": boxes})

## B6 is not run

Grounding DINO + SAM 2.1 (our M2 teacher) on the Open Images masks is Should only (ruling R22): it needs notebook 04's
stack for 284 photos, and M2 already reports the teacher's by-eye quality (84% good of 491).
`serpentguard.baselines.grounding_dino_sam_baseline` raises `NotImplementedError` with that reason; it can be added
in M4.

In [17]:
if E2_RUN_DIR is not None:
    parts = [np.load(Path(E2_RUN_DIR) / name) for name in ("val_outputs.npz", "calib_outputs.npz")]
    pool = pd.DataFrame({"path": np.concatenate([part["paths"] for part in parts]),
                         "class_id": np.concatenate([part["labels"] for part in parts])})
    write_json(FEWSHOT_OUT / "C1_E2.json", tier2_curve(np.concatenate([part["embedding"] for part in parts]), pool,
                                                       f"E2 embeddings from {Path(E2_RUN_DIR).name}"))
else:
    print("E2_RUN_DIR is None: C1 on E2 features waits for an E2 run")

E2_RUN_DIR is None: C1 on E2 features waits for an E2 run


In [18]:
PROTOTYPE_KINDS = ("plain", "balanced")

if K1_RUN_DIR is not None:
    run_dir = Path(K1_RUN_DIR)
    with np.load(run_dir / "heldout_outputs.npz") as heldout, np.load(run_dir / "prototypes.npz") as prototypes, \
            np.load(run_dir / "val_outputs.npz") as val_old:
        heldout, val_old = dict(heldout), dict(val_old)
        enrolment = {kind: enrolment_eval(heldout, prototypes[kind], val_old) for kind in PROTOTYPE_KINDS}
    write_json(FEWSHOT_OUT / "K1_enrolment.json", {"run": run_dir.name, **enrolment})
    for kind, result in enrolment.items():
        for shots, scores in result["shots"].items():
            print(f"K1 {kind}, {shots} shots: new top-1 {scores['new_top1']:.3f} +/- {scores['new_top1_std']:.3f} "
                  f"(other photographers {scores['new_top1_other_photographer']:.3f}), called an old class "
                  f"{scores['new_as_old']:.3f}; old top-1 {result['old_top1_before']:.3f} -> "
                  f"{scores['old_top1_after']:.3f}")
else:
    print("K1_RUN_DIR is None: K1 enrolment waits for the K1 run")

K1_RUN_DIR is None: K1 enrolment waits for the K1 run


In [19]:
rows = [summary_row(path.parent.name, json.loads(path.read_text()))
        for path in sorted(OUT.glob("*/final_metrics.json"))]
comparison = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
comparison.to_csv(OUT / "comparison.csv")
missing = [baseline for baseline in RUN_BASELINES if baseline != "C1" and not any(
    row["model"].startswith(baseline) for row in rows)]
assert not missing, f"baselines without a final_metrics.json: {missing}"
print(comparison.round(3).to_string())

AssertionError: baselines without a final_metrics.json: ['B1']